In [1]:
import pandas as pd
import numpy as np
import swifter

In [2]:
df = pd.read_csv("loadboard_full.csv")
df.shape

C:\Users\kgome\AppData\Local\Temp\ipykernel_29756\3804631142.py:1: DtypeWarning: Columns (17,18,20) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("loadboard_full.csv")


(1581547, 22)

In [3]:
dic = {'Distance_mi':'Distance', 'TruckType':'Equip', 'File_Date':'Removed','File':'Source', 'Rate_mi':'RatePerMile', 'Weight_lbs':'Weight','FirstActiveDate':'Posted'}
df = df.rename(columns=dic)

In [4]:
print(df["Posted"].min())
print(df["Posted"].max())

2022-06-28
2023-04-14


In [5]:
cols=['ID', 'Posted', 'PickUp', 'CityOrigin', 'LatOrigin', 'LngOrigin', 'CityDestination',
      'LatDestination', 'LngDestination', 'Distance', 'Source', 'Equip','StateOrigin',
      'HubOrigin', 'StateDestination', 'HubDestination','Broker_Shipper',"Weight"]

In [6]:
#borrar duplicados entre loadboards

def remove_duplicates_general(data: pd.DataFrame) -> pd.DataFrame:
    """
    This function removes duplicate rows from the provided DataFrame
    based on specific columns.
    Args:
    data (pd.DataFrame): The dataset to be processed.
    Returns:
    pd.DataFrame: A DataFrame with duplicates removed.
    """
    # Ensure the input is a DataFrame
    if not isinstance(data, pd.DataFrame):
        raise ValueError("Input data must be a pandas DataFrame.")
    # List of columns to check for duplicates
    cols_duplicates = [
       "Posted",
        "CityOrigin",
        "StateOrigin",
        "CityDestination",
        "StateDestination",
        "PickUp",
        "Equip",
        "Broker_Shipper",
        "Weight",
        'Distance',
        #'RatePerMile',
       # "Size",
    ]
    # Ensure all specified columns exist in the DataFrame
    missing_cols = [col for col in cols_duplicates if col not in data.columns]
    if missing_cols:
        raise ValueError(
            f"The following columns are missing from the DataFrame: {', '.join(missing_cols)}"
        )
    # Remove duplicates based on the specified columns
    df = data.drop_duplicates(
        subset=cols_duplicates, keep="first"
    )  # Keep the first occurrence
    return df

df_removed_dupl = remove_duplicates_general(df)

In [7]:
df.shape[0] - df_removed_dupl.shape[0]

22873

In [8]:
df_removed_dupl['RatePerMile'] = df_removed_dupl['RatePerMile'].replace({'x':np.nan})
df_removed_dupl['RatePerMile'] = pd.to_numeric(df_removed_dupl['RatePerMile'], errors='coerce')
df_removed_dupl.loc[df_removed_dupl['Rate_Unit'] == 'x', 'RatePerMile'] = (df_removed_dupl['RatePerMile'] / df_removed_dupl['Distance'])

C:\Users\kgome\AppData\Local\Temp\ipykernel_29756\2440204990.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_removed_dupl['RatePerMile'] = df_removed_dupl['RatePerMile'].replace({'x':np.nan})
C:\Users\kgome\AppData\Local\Temp\ipykernel_29756\2440204990.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_removed_dupl['RatePerMile'] = pd.to_numeric(df_removed_dupl['RatePerMile'], errors='coerce')


In [9]:
df_removed_dupl.drop(df_removed_dupl[(df_removed_dupl['Rate_Unit'] == 'mi') & 
                                   (df_removed_dupl['RatePerMile'] > 50) & 
                                   (df_removed_dupl['Distance'] > 50)].index, inplace=True)

C:\Users\kgome\AppData\Local\Temp\ipykernel_29756\2247494706.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_removed_dupl.drop(df_removed_dupl[(df_removed_dupl['Rate_Unit'] == 'mi') &


In [10]:
df_removed_dupl.drop(df_removed_dupl[(df_removed_dupl['Rate_Unit'] == 'mi') & 
                                   (df_removed_dupl['RatePerMile'] > 100)].index, inplace=True)

C:\Users\kgome\AppData\Local\Temp\ipykernel_29756\1240280340.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_removed_dupl.drop(df_removed_dupl[(df_removed_dupl['Rate_Unit'] == 'mi') &


In [11]:
#ver si esto realmente ocurre
#ciudades origen/destino desconocidas

def remove_empty_values(df: pd.DataFrame) -> pd.DataFrame:
    """
    Removes rows from the DataFrame where any of the following columns contain empty or missing
    values: 'StateOrigin','StateDestination', 'CityOrigin', 'CityDestination'

    This function filters out rows with NaN or empty strings ('') in the specified columns to
    ensure that the data is complete for further processing.

    Args:
        df (pd.DataFrame): The input DataFrame containing the columns to be checked for missing
        or empty values.

    Returns:
        pd.DataFrame: The filtered DataFrame with rows removed where any of the specified columns
        have missing or empty values.
    """

    query_condition = (
        "StateOrigin.notna() & StateDestination.notna() & "
        "CityOrigin.notna() & CityDestination.notna()"
    )

    second_condition = (
        "StateOrigin != '' & StateDestination != '' & "
        "CityOrigin != '' & CityDestination != ''"
    )

    filtered_df = df.query(query_condition).query(second_condition)

    return filtered_df

df_cities_nans = remove_empty_values(df_removed_dupl)

In [12]:
df_removed_dupl.shape[0] - df_cities_nans.shape[0]

0

In [13]:
df_removed_dupl.to_csv('dfaugustoready.csv')

In [14]:
#df_removed_dupl para el análisis de clusters
#df_cities_nans para el EDA

### 1. Obtener coordenadas

In [15]:
import pandas as pd
df_removed_dupl = pd.read_csv('dfaugustoready.csv')

C:\Users\kgome\AppData\Local\Temp\ipykernel_29756\4096750906.py:2: DtypeWarning: Columns (18,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df_removed_dupl = pd.read_csv('dfaugustoready.csv')


In [16]:
origen = df_removed_dupl[['CityOrigin', 'StateOrigin']].drop_duplicates().rename(columns={'CityOrigin':'city','StateOrigin':'state'})
destino = df_removed_dupl[['CityDestination', 'StateDestination']].drop_duplicates().rename(columns={'CityDestination':'city','StateDestination':'state'})
ciudades_estados = pd.concat([origen, destino]).drop_duplicates().reset_index(drop=True)
len(ciudades_estados)

24971

In [17]:
ciudades_estados

,city,state
0,Coquille,OR
1,Victorville,CA
2,Stockton,CA
3,Olivehurst,CA
4,Moreno Valley,CA
...,...,...
24966,Belvedere Park,GA
24967,Eastland,TX
24968,Oakland,IL
24969,Arnolds Park,IA


In [18]:
import sys
sys.path.append("../utils")

from coord import *
from hubs_coordpt2 import *

In [19]:
new_columns = ['lat', 'lng']

In [20]:
import numpy as np
for col in new_columns:
    if col not in ciudades_estados.columns:
        ciudades_estados[col] = np.nan

In [21]:
ciudades_estados

,city,state,lat,lng
0,Coquille,OR,NaN,NaN
1,Victorville,CA,NaN,NaN
2,Stockton,CA,NaN,NaN
3,Olivehurst,CA,NaN,NaN
4,Moreno Valley,CA,NaN,NaN
...,...,...,...,...
24966,Belvedere Park,GA,NaN,NaN
24967,Eastland,TX,NaN,NaN
24968,Oakland,IL,NaN,NaN
24969,Arnolds Park,IA,NaN,NaN


In [23]:
#ciudades_estados = ciudades_estados.apply(process_row, axis=1)
#ciudades_estados.head()

In [ ]:
#ciudades_estados.loc[(ciudades_estados['state']=='DC'),'city']='Washington'
#ciudades_estados.loc[(ciudades_estados['state']=='DC'),'lat']=38.9047
#ciudades_estados.loc[(ciudades_estados['state']=='DC'),'lng']=-77.0163
#ciudades_estados.loc[(ciudades_estados['state']=='DC'),'state']='WA'

In [ ]:
#ciudades_estados.drop(ciudades_estados.loc[ciudades_estados['state'].isin(['ON', 'MB', 'NL', 'PU', 'QC', 'PUE'])].index,inplace=True)
#ciudades_estados.shape[0]

In [ ]:
#ciudades_estados.to_csv('coordenadas_123.csv')

In [27]:
ciudades_estados = pd.read_csv('coordenadas_123.csv')
ciudades_estados.info()
#ciudades_estados es la lista de coordenadas que se usan para generar los hubs

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24820 entries, 0 to 24819
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Unnamed: 0.1  24820 non-null  int64  
 1   Unnamed: 0    24820 non-null  int64  
 2   city          24820 non-null  object 
 3   state         24820 non-null  object 
 4   lat           15173 non-null  float64
 5   lng           15173 non-null  float64
 6   near_hub      15173 non-null  object 
dtypes: float64(2), int64(2), object(3)
memory usage: 1.3+ MB


#### Agregar los hubs

In [26]:
#from hubs_coordpt2 import hub_assignment
#hub_assign = pd.read_csv('../utils/hubs_assign.csv')
#hubs_123 = hub_assignment(ciudades_estados,hub_assign,'hubs_id.xlsx')
#ciudades_estados.to_csv('coordenadas_123.csv')
#en este paso a cada coordenada se le asigna el hub correspondiente 

In [ ]:
hubs123 = pd.read_csv('coordenadas_123.csv')
hubs123

### Agregar información al df final

In [ ]:
df_removed_dupl.drop(df_removed_dupl.loc[df_removed_dupl['StateOrigin'].isin(['ON', 'MB', 'NL', 'PU', 'QC', 'PUE'])].index,inplace=True)
df_removed_dupl.drop(df_removed_dupl.loc[df_removed_dupl['StateDestination'].isin(['ON', 'MB', 'NL', 'PU', 'QC', 'PUE'])].index,inplace=True)
df_removed_dupl.shape[0]

In [ ]:
df_coord = pd.merge(df_removed_dupl,
            hubs123,
            left_on=["CityOrigin",'StateOrigin'],
            right_on=['city','state'],
            how="left",
        ).rename(columns={'lat':'LatOrigin','lng':'LngOrigin','near_hub':'HubOrigin'}).drop(['city','state','Unnamed: 0'],axis=1)

In [ ]:
df_coord = pd.merge(df_coord,
            hubs123,
            left_on=["CityDestination",'StateDestination'],
            right_on=['city','state'],
            how="left",
        ).rename(columns={'lat':'LatDestination','lng':'LngDestination','near_hub':'HubDestination'}).drop(['city','state','Unnamed: 0'],axis=1)

In [ ]:
df_coord.head()

In [36]:
df_coord.to_csv('123oldloadsready.csv')
#a diferencia de coordenadas_123 este es la loadboard como tal con las cargas,
#pero cada carga asignada a los hubs del método actual para posteriores comparaciones